# Feature Engineering

Nesta etapa serão realizados os tratamentos necessários para preparar os dados para a modelagem.

Os principais objetivos são:

- Tratar valores ausentes;
- Remover variáveis que não serão utilizadas;
- Corrigir tipos de dados;
- Codificar variáveis categóricas;
- Avaliar multicolinearidade;
- Selecionar as variáveis explicativas;
- Gerar a base final para treinamento dos modelos.

In [15]:
import pandas as pd
import numpy as np
import openpyxl

In [2]:
df = pd.read_excel('../data/raw/Base_Case_Modelagem2.xlsx', sheet_name='Base de Treino')

In [3]:
df.head(1)

,ID,PERFORMANCE (var resposta),V1,V2,V3,V4,V5,V6,V7,V8,...,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19
0,1,0.0,563.10199,B,12.0,6.0,1264.109985,1.67665,944.123108,0.0375,...,10.0,353.0,Fiel,Fiel,Fiel,0.0,1.0,Premium,0.74687,49


In [4]:
df.drop(columns=['ID'], inplace=True)

In [5]:
df = df[df['PERFORMANCE (var resposta)'].notna()].copy()

In [6]:
df.shape

(18745, 20)

In [7]:
df['PERFORMANCE (var resposta)'].isnull().sum()

np.int64(0)

In [14]:
X = df.drop(columns=['PERFORMANCE (var resposta)'])

y = df['PERFORMANCE (var resposta)']

In [16]:
from sklearn.impute import SimpleImputer

num_cols = X.select_dtypes(include=np.number).columns
print(num_cols)

Index(['V1', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V15',
       'V16', 'V18', 'V19'],
      dtype='object')


In [17]:
imputer_num = SimpleImputer(strategy='median')

In [18]:
X[num_cols] = imputer_num.fit_transform(X[num_cols])

In [20]:
cat_cols = X.select_dtypes(include="object").columns
imputer_cat = SimpleImputer(strategy='most_frequent')
X[cat_cols] = imputer_cat.fit_transform(X[cat_cols])

In [21]:
X.isnull().sum()

V1     0
V2     0
V3     0
V4     0
V5     0
V6     0
V7     0
V8     0
V9     0
V10    0
V11    0
V12    0
V13    0
V14    0
V15    0
V16    0
V17    0
V18    0
V19    0
dtype: int64

In [22]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)

resultado = encoder.fit_transform(X[cat_cols])


In [23]:
X_cat = pd.DataFrame(
    resultado,
    columns=encoder.get_feature_names_out(cat_cols),
    index=X.index
)

In [24]:
X = pd.concat([X.drop(columns=cat_cols), X_cat], axis=1)

In [30]:
X.shape


(18745, 41)

In [ ]:
X.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18745 entries, 0 to 35571
Data columns (total 41 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   V1                18745 non-null  float64
 1   V3                18745 non-null  float64
 2   V4                18745 non-null  float64
 3   V5                18745 non-null  float64
 4   V6                18745 non-null  float64
 5   V7                18745 non-null  float64
 6   V8                18745 non-null  float64
 7   V9                18745 non-null  float64
 8   V10               18745 non-null  float64
 9   V11               18745 non-null  float64
 10  V15               18745 non-null  float64
 11  V16               18745 non-null  float64
 12  V18               18745 non-null  float64
 13  V19               18745 non-null  float64
 14  V2_B              18745 non-null  float64
 15  V2_G              18745 non-null  float64
 16  V2_M              18745 non-null  float64
 17

In [34]:
for col in X.columns:
    if X[col].dtype == 'float64':
        print(f"{col}: {X[col].nunique()} unique values")

V1: 17739 unique values
V3: 77 unique values
V4: 65 unique values
V5: 11135 unique values
V6: 2177 unique values
V7: 11482 unique values
V8: 4930 unique values
V9: 8241 unique values
V10: 91 unique values
V11: 461 unique values
V15: 3 unique values
V16: 2 unique values
V18: 10623 unique values
V19: 76 unique values
V2_B: 2 unique values
V2_G: 2 unique values
V2_M: 2 unique values
V2_W: 2 unique values
V12_Ausente: 2 unique values
V12_Fiel: 2 unique values
V12_Inativo: 2 unique values
V12_Novo: 2 unique values
V12_Ocasional: 2 unique values
V13_Ausente: 2 unique values
V13_Fiel: 2 unique values
V13_Inativo: 2 unique values
V13_Novo: 2 unique values
V13_Ocasional: 2 unique values
V14_Ausente: 2 unique values
V14_Fiel: 2 unique values
V14_Inativo: 2 unique values
V14_Novo: 2 unique values
V14_Ocasional: 2 unique values
V17_Ativo: 2 unique values
V17_Ausente: 2 unique values
V17_Basico: 2 unique values
V17_Conveniencia: 2 unique values
V17_Economico: 2 unique values
V17_Inativo: 2 unique v

In [38]:
corr = X.corr(numeric_only=True).abs()

In [39]:
upper = corr.where(
    np.triu(
        np.ones(corr.shape),
        k=1
    ).astype(bool)
)

In [40]:
to_drop = [
    column
    for column in upper.columns
    if any(upper[column] > 0.90)
]

to_drop

['V7', 'V17_Ausente', 'V17_Inativo']

In [41]:
corr["V7"].sort_values(ascending=False).head(10)

V7             1.000000
V5             0.934534
V18            0.566382
V6             0.560907
V4             0.475139
V12_Fiel       0.454118
V17_Premium    0.430578
V13_Fiel       0.396320
V10            0.355995
V13_Inativo    0.354955
Name: V7, dtype: float64

In [42]:
corr_target = df.corr(numeric_only=True)["PERFORMANCE (var resposta)"]

corr_target[["V5", "V7"]]

V5   -0.113540
V7   -0.117396
Name: PERFORMANCE (var resposta), dtype: float64

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf.fit(X, y)